# Clase 2 — Bases de datos vectoriales (vector stores)

## De qué se trata

En la Clase 1 convertiste texto en embeddings y comparaste dos vectores a mano con similitud coseno. Ahora escalamos esa idea: en vez de comparar la pregunta contra 3 o 4 chunks con un `for`, un **vector store** (Pinecone, Chroma, FAISS, pgvector...) guarda miles o millones de embeddings y te devuelve los más parecidos en milisegundos.

Vamos a construir, paso a paso, el vector store de **SoporteBot**: un chatbot que responde preguntas de usuarios sobre wifi, facturación, seguridad y RRHH buscando en una base de FAQs (el mismo dataset que usaste en `extras/embeddings`).

## 1. Qué guarda un vector store

No alcanza con guardar el vector. Si SoporteBot solo tuviera números, no podría mostrarle al usuario ni el texto de la respuesta ni de qué FAQ salió. Por eso cada fila del vector store guarda:

| Campo | Ejemplo en SoporteBot | Para qué sirve |
|---|---|---|
| chunk_id | `red-002` | Identifica el fragmento para trazabilidad |
| embedding | `[0.12, -0.04, ...]` | Es lo que se compara al buscar |
| content | "Reinicia el router 30 segundos" | El texto que va a leer el LLM |
| metadata | `{"source": "manual_red.md"}` | Permite filtrar y citar la fuente |
| configuración del índice | modelo de embeddings, métrica | Hace que la búsqueda sea reproducible |

In [ ]:
# El camino que recorre una pregunta de un usuario hasta la respuesta del chatbot.
import matplotlib.pyplot as plt

pasos = ["pregunta del usuario", "embedding", "vector store", "Top-K chunks", "respuesta del LLM"]
plt.figure(figsize=(9, 2.3))
plt.plot(range(len(pasos)), [0] * len(pasos), "o-", color="#1f4e79")
for i, paso in enumerate(pasos):
    plt.text(i, .1, paso, ha="center")
plt.axis("off")
plt.title("De la pregunta a la respuesta: donde entra el vector store")
plt.show()

## 2. Búsqueda exacta vs. aproximada

Con 3 FAQs, comparar la pregunta contra todas es instantáneo (eso hiciste en `extras/embeddings`). Pero si SoporteBot tuviera 500.000 FAQs de una empresa grande, comparar contra cada una en cada pregunta sería lento. Ahí aparecen los índices **ANN** (Approximate Nearest Neighbors, "vecinos más cercanos aproximados"): sacrifican un poco de precisión para responder mucho más rápido.

| Estrategia | ¿Qué tan exacta es? | ¿Qué tan rápida a gran escala? | Cuándo la usás |
|---|---|---|---|
| k-NN exacto | Siempre encuentra el mejor resultado | Se vuelve lenta con muchos documentos | Pocos documentos, o para verificar que el ANN no se equivoca |
| HNSW | Muy buena, a cambio de más memoria | Rápida | Millones de vectores, búsquedas frecuentes |
| IVF | Se ajusta con un parámetro (`nprobe`) | Rápida | Corpus grandes con presupuesto de memoria más chico |

In [ ]:
# A medida que crece la base de FAQs, comparar "contra todos" (exacto) se vuelve mas caro.
import matplotlib.pyplot as plt

cantidad_faqs = [100, 1_000, 10_000, 100_000]
trabajo_exacto = [1, 10, 100, 1000]
trabajo_ann = [1, 3, 8, 20]

plt.plot(cantidad_faqs, trabajo_exacto, "o-", label="busqueda exacta")
plt.plot(cantidad_faqs, trabajo_ann, "s-", label="busqueda ANN")
plt.xscale("log")
plt.xlabel("cantidad de FAQs indexadas")
plt.ylabel("trabajo relativo por pregunta")
plt.title("Por que los chatbots grandes usan ANN")
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 3. Normalización y qué significa el score

Antes de indexar los embeddings de tus FAQs, revisá: que todos tengan la misma cantidad de dimensiones, que ninguno sea un vector de puros ceros, y que uses el **mismo modelo de embeddings** para las FAQs y para la pregunta del usuario (si mezclás modelos, los vectores no son comparables entre sí).

> Ojo con el sentido del score: con similitud coseno o producto interno, **más alto es mejor**. Con distancia L2 (euclídea), es al revés: **más bajo es mejor**. Confundir esto hace que tu chatbot muestre la peor FAQ como si fuera la mejor.

## Ejemplo: Top-K y Recall@K en SoporteBot

Supongamos que SoporteBot ya calculó los scores de similitud entre la pregunta de un usuario y 5 FAQs. Con el código de abajo sacamos el Top-3 exacto y lo comparamos contra lo que devolvió (a modo de ejemplo) un índice ANN, para ver qué tan bien se pareció.

In [ ]:
import numpy as np

def top_k(scores, k=3):
    """Devuelve los indices de los k scores mas altos."""
    return np.argsort(scores)[::-1][:k].tolist()

# Scores de similitud entre la pregunta del usuario y 5 FAQs (inventados para el ejemplo).
faqs = ["wifi", "facturacion", "contrasena", "horarios", "envios"]
scores = np.array([.34, .91, .72, .18, .66])

top_exacto = top_k(scores, k=3)
top_ann = [1, 2, 4]  # lo que devolvio (a modo de ejemplo) un indice ANN

recall_at_3 = len(set(top_exacto) & set(top_ann)) / len(top_exacto)

print("Top-3 exacto:", [faqs[i] for i in top_exacto])
print("Top-3 ANN:", [faqs[i] for i in top_ann])
print("Recall@3:", round(recall_at_3, 2), "-> de cada 3 FAQs correctas, cuantas encontro el ANN")

## Cómo documentar un experimento

Cuando pruebes un índice nuevo para SoporteBot (por ejemplo, pasar de búsqueda exacta a FAISS), anotá esto para poder comparar de forma justa más adelante:

| Backend | Modelo | Métrica | Top-K | Recall@K | p50 | p95 | Memoria |
|---|---|---|---:|---:|---:|---:|---:|
| Exacto | registrar | coseno/IP | 5 | 1.00 | medir | medir | medir |
| ANN | registrar | igual | 5 | medir | medir | medir | medir |

## Errores comunes al armar un vector store

- Copiar un valor de configuración (chunk size, k, métrica) sin probarlo contra tus propias FAQs.
- Confundir "la demo funcionó una vez" con "medí que funciona bien" — para eso está Recall@K.
- No guardar el `chunk_id`, la fuente o la configuración del índice: después no podés explicar por qué el chatbot respondió eso.